# 05 Generate Prediction Lookup

Pre-compute typical zone/hour/day demand predictions for the Phase 2 Live Prediction page.

## Section 0: Imports

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


## Section 1: Load Trained Model and Data

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = ROOT / "models" / "adaptive_model_final.pkl"
FEATURE_COLUMNS_PATH = ROOT / "models" / "feature_columns.json"
WEEKLY_PREDICTIONS_PATH = ROOT / "data" / "processed" / "weekly_predictions.csv"
ZONES_PATH = ROOT / "frontend" / "public" / "data" / "zones.json"
OUTPUT_PATH = ROOT / "frontend" / "public" / "data" / "prediction_lookup.json"

model = joblib.load(MODEL_PATH)
feature_columns = json.loads(FEATURE_COLUMNS_PATH.read_text())
history = pd.read_csv(WEEKLY_PREDICTIONS_PATH, parse_dates=["timestamp"])
zones_payload = json.loads(ZONES_PATH.read_text())
zones = pd.DataFrame(zones_payload["zones"]).sort_values("rank").head(20)


## Section 2: Build Prediction Scenarios

Lag and rolling features are reconstructed from historical hourly demand, then averaged by zone, hour, and day of week.

In [ ]:
history = history[history["zone_id"].isin(zones["id"])].copy().sort_values(["zone_id", "timestamp"])
history["hour"] = history["timestamp"].dt.hour
history["day_of_week"] = history["timestamp"].dt.dayofweek
history["day_of_month"] = history["timestamp"].dt.day
history["month"] = history["timestamp"].dt.month
history["is_weekend"] = history["day_of_week"].isin([5, 6]).astype(int)
history["is_rush_hour"] = history["hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
history["hour_sin"] = np.sin(2 * np.pi * history["hour"] / 24)
history["hour_cos"] = np.cos(2 * np.pi * history["hour"] / 24)
history["dow_sin"] = np.sin(2 * np.pi * history["day_of_week"] / 7)
history["dow_cos"] = np.cos(2 * np.pi * history["day_of_week"] / 7)
history["borough_Manhattan"] = (history["borough"] == "Manhattan").astype(int)
history["borough_Queens"] = (history["borough"] == "Queens").astype(int)

grouped = history.groupby("zone_id", group_keys=False)
for lag in [1, 2, 3, 24, 168]:
    history[f"lag_{lag}h"] = grouped["actual_demand"].shift(lag)

shifted = grouped["actual_demand"].shift(1)
history["rolling_mean_6h"] = shifted.groupby(history["zone_id"]).rolling(6, min_periods=1).mean().reset_index(level=0, drop=True)
history["rolling_mean_24h"] = shifted.groupby(history["zone_id"]).rolling(24, min_periods=1).mean().reset_index(level=0, drop=True)
history["rolling_mean_7d"] = shifted.groupby(history["zone_id"]).rolling(168, min_periods=1).mean().reset_index(level=0, drop=True)
history["rolling_std_24h"] = shifted.groupby(history["zone_id"]).rolling(24, min_periods=2).std().reset_index(level=0, drop=True)
history["rolling_max_24h"] = shifted.groupby(history["zone_id"]).rolling(24, min_periods=1).max().reset_index(level=0, drop=True)
history["rolling_min_24h"] = shifted.groupby(history["zone_id"]).rolling(24, min_periods=1).min().reset_index(level=0, drop=True)

for column in feature_columns:
    if column in history.columns and history[column].isna().any():
        history[column] = history[column].fillna(history.groupby("zone_id")[column].transform("median"))
        history[column] = history[column].fillna(history[column].median())

fixed_columns = {"hour", "day_of_week", "is_weekend", "is_rush_hour", "hour_sin", "hour_cos", "dow_sin", "dow_cos", "zone_id", "borough_Manhattan", "borough_Queens"}
average_columns = [column for column in feature_columns if column not in fixed_columns]
pattern_means = history.groupby(["zone_id", "hour", "day_of_week"])[average_columns + ["actual_demand"]].mean()
pattern_std = history.groupby(["zone_id", "hour", "day_of_week"])["actual_demand"].std()
zone_hour_means = history.groupby(["zone_id", "hour"])[average_columns + ["actual_demand"]].mean()
zone_means = history.groupby("zone_id")[average_columns + ["actual_demand"]].mean()
global_means = history[average_columns + ["actual_demand"]].mean()
zone_quantiles = history.groupby("zone_id")["actual_demand"].quantile([0.25, 0.75, 0.95]).unstack()
zone_std = history.groupby("zone_id")["actual_demand"].std()


## Section 3: Generate Feature Vectors

In [ ]:
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def get_typical_values(zone_id, hour, dow):
    if (zone_id, hour, dow) in pattern_means.index:
        return pattern_means.loc[(zone_id, hour, dow)]
    if (zone_id, hour) in zone_hour_means.index:
        return zone_hour_means.loc[(zone_id, hour)]
    if zone_id in zone_means.index:
        return zone_means.loc[zone_id]
    return global_means

scenarios = []
for zone in zones.to_dict("records"):
    zone_id = int(zone["id"])
    for hour in range(24):
        for dow in range(7):
            typical = get_typical_values(zone_id, hour, dow)
            row = {column: 0.0 for column in feature_columns}
            for column in average_columns:
                row[column] = float(typical[column]) if column in typical and pd.notna(typical[column]) else float(global_means.get(column, 0.0))
            row.update({
                "hour": hour,
                "day_of_week": dow,
                "day_of_month": int(round(row.get("day_of_month", 15))) or 15,
                "month": int(round(row.get("month", 3))) or 3,
                "is_weekend": int(dow in [5, 6]),
                "is_rush_hour": int(hour in [7, 8, 9, 16, 17, 18, 19]),
                "hour_sin": float(np.sin(2 * np.pi * hour / 24)),
                "hour_cos": float(np.cos(2 * np.pi * hour / 24)),
                "dow_sin": float(np.sin(2 * np.pi * dow / 7)),
                "dow_cos": float(np.cos(2 * np.pi * dow / 7)),
                "zone_id": zone_id,
                "borough_Manhattan": int(zone["borough"] == "Manhattan"),
                "borough_Queens": int(zone["borough"] == "Queens"),
            })
            scenarios.append({"zone": zone, "hour": hour, "dow": dow, "features": row, "historical_avg": float(typical["actual_demand"])})

features = pd.DataFrame([scenario["features"] for scenario in scenarios], columns=feature_columns)
features.head()


## Section 4: Generate Predictions

In [ ]:
predicted = np.maximum(model.predict(features), 0)


## Section 5: Generate Comparison Predictions

In [ ]:
def demand_category(zone_id, prediction):
    q25, q75, q95 = zone_quantiles.loc[zone_id, [0.25, 0.75, 0.95]]
    if prediction < q25:
        return "Low"
    if prediction < q75:
        return "Medium"
    if prediction < q95:
        return "High"
    return "Very High"


## Section 6: Generate Explanation Data

In [ ]:
feature_importance = dict(zip(feature_columns, getattr(model, "feature_importances_", np.ones(len(feature_columns)))))

def explanation(zone, hour, dow, category):
    lag_importance = sum(feature_importance.get(name, 0) for name in ["lag_1h", "lag_24h", "lag_168h", "rolling_mean_24h", "rolling_mean_7d"])
    hour_importance = feature_importance.get("hour", 0) + feature_importance.get("is_rush_hour", 0)
    day_importance = feature_importance.get("day_of_week", 0) + feature_importance.get("dow_sin", 0) + feature_importance.get("dow_cos", 0)

    if lag_importance > hour_importance + day_importance:
        first = "Recent trend from lagged demand signals"
    elif hour in [7, 8, 9, 16, 17, 18, 19]:
        first = "Peak rush hour traffic"
    elif hour <= 5:
        first = "Late night - typically quieter period"
    else:
        first = "Hour-of-day demand pattern"

    factors = [
        first,
        "Weekend pattern - typically lower commuter demand" if dow >= 5 else f"{day_names[dow]} weekday pattern",
        "Airport zone - consistently high all day" if zone["borough"] == "Queens" and "Airport" in zone["name"] else ("Manhattan baseline activity" if zone["borough"] == "Manhattan" else f"{zone['borough']} local demand baseline"),
    ]
    if category == "Very High" and first != "Peak rush hour traffic":
        factors[0] = "Very high relative to this zone's history"
    return factors[:3]


## Section 7: Export to JSON

In [ ]:
records = {}
for scenario, value in zip(scenarios, predicted):
    zone = scenario["zone"]
    zone_id = int(zone["id"])
    hour = scenario["hour"]
    dow = scenario["dow"]
    std = pattern_std.get((zone_id, hour, dow), np.nan)
    if pd.isna(std) or std == 0:
        std = zone_std.get(zone_id, history["actual_demand"].std())
    historical_avg = scenario["historical_avg"]
    pct = 0.0 if historical_avg == 0 else ((float(value) - historical_avg) / historical_avg) * 100
    category = demand_category(zone_id, float(value))
    records[f"{zone_id}_{hour}_{dow}"] = {
        "zone_id": zone_id,
        "zone_name": zone["name"],
        "borough": zone["borough"],
        "hour": hour,
        "day_of_week": dow,
        "day_name": day_names[dow],
        "predicted_demand": round(float(value), 1),
        "confidence_lower": round(max(0.0, float(value) - 1.96 * float(std)), 1),
        "confidence_upper": round(float(value) + 1.96 * float(std), 1),
        "historical_avg": round(float(historical_avg), 1),
        "vs_historical_pct": round(float(pct), 1),
        "demand_category": category,
        "explanation_factors": explanation(zone, hour, dow, category),
    }

payload = {
    "metadata": {
        "model_version": "adaptive_v9",
        "generated_at": pd.Timestamp.now(tz="America/New_York").isoformat(),
        "scenarios_count": len(records),
        "model_smape": 22.78,
    },
    "predictions": records,
}
OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


## Section 8: Verification

In [ ]:
print(list(payload["predictions"].items())[:5])
assert payload["metadata"]["scenarios_count"] == 3360
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")
print("? Prediction lookup table generated")
